#Imports and Uploading the Dataset

In [22]:
# The -q flag keeps it "quiet" so it doesn't crash your browser printing thousands of image names
!unzip -q "/content/Dataset.zip" -d "/content/Dataset"

print("✅ Dataset successfully extracted to local SSD!")

✅ Dataset successfully extracted to local SSD!


In [23]:
!pip install -q transformers timm

In [24]:
import os
import zipfile
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import convnext_base, ConvNeXt_Base_Weights
from transformers import ViTModel
from PIL import Image
from google.colab import drive

In [44]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
CSV_PATH = '/content/Dataset/Dataset/Labels/labels.csv'
EXTRACT_DIR = '/content/Dataset/Dataset/raw_crash_images'

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


#Data Loading Pipeline

In [27]:
EXPECTED_PARTS = [
    "front_bumper", "rear_bumper", "hood", "trunk", "roof",
    "windshield", "rear_window", "left_front_fender", "right_front_fender",
    "left_rear_quarter_panel", "right_rear_quarter_panel", "left_front_door",
    "right_front_door", "left_rear_door", "right_rear_door",
    "headlights", "taillights", "grille"
]

In [28]:
class CarDamageDataset(Dataset):
  def __init__(self, csv_file, img_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

  def __len__(self):
        return len(self.data_frame)

  def __getitem__(self, idx):
        try:
            # 1. Load the Image
            img_name = os.path.join(self.img_dir, self.data_frame.iloc[idx]['filename'])
            image = Image.open(img_name).convert('RGB')

            if self.transform:
                image = self.transform(image)

            # 2. Extract the 18 labels
            labels = {}
            for part in EXPECTED_PARTS:
                labels[part] = torch.tensor(self.data_frame.iloc[idx][part], dtype=torch.long)

            return image, labels

        except Exception as e:
            print(f"\n[WARNING] Skipping corrupt image: {img_name}. Error: {e}")
            # If an image is broken, safely load the very first image in the dataset instead to keep the batch size intact
            return self.__getitem__(0)

##Augmentation Pipeline

In [29]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Standard ImageNet normalization
])

##Build the Dataset

In [30]:
full_dataset = CarDamageDataset(csv_file=CSV_PATH, img_dir=EXTRACT_DIR, transform=transform)

##Train-Val-Split



In [31]:
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

##Data Loaders

In [32]:
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [33]:
print("PyTorch DataLoaders constructed successfully.")

PyTorch DataLoaders constructed successfully.


#Constructing Neural Network

In [34]:
class DualStreamHydra(nn.Module):
    def __init__(self, num_classes=4):
        super(DualStreamHydra, self).__init__()

        # --- STREAM 1: ConvNeXt (Local Feature Extractor) ---
        print("Loading ConvNeXt Base Stream...")
        convnext = convnext_base(weights=ConvNeXt_Base_Weights.IMAGENET1K_V1)
        self.cnn_features = convnext.features
        self.cnn_pool = convnext.avgpool

        # --- STREAM 2: Vision Transformer (Global Context) ---
        print("Loading ViT Stream...")
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")

        # Freeze BOTH backbones for Phase 1
        for param in self.cnn_features.parameters():
            param.requires_grad = False
        for param in self.vit.parameters():
            param.requires_grad = False

        self.dropout = nn.Dropout(0.4)

        # --- THE FUSION HEADS ---
        # ConvNeXt outputs 1024. ViT outputs 768. Combined = 1792.
        self.heads = nn.ModuleDict({
            part: nn.Linear(1792, num_classes) for part in EXPECTED_PARTS
        })

    def forward(self, x):
        # 1. Process Stream A (CNN)
        cnn_out = self.cnn_features(x)
        cnn_out = self.cnn_pool(cnn_out)
        cnn_context = torch.flatten(cnn_out, 1) # Shape: (Batch, 1024)

        # 2. Process Stream B (ViT)
        # We pass the raw image directly to the ViT, bypassing the Hugging Face error completely
        vit_out = self.vit(pixel_values=x)
        vit_context = vit_out.pooler_output # Shape: (Batch, 768)

        # 3. Feature Fusion (Concatenation)
        fused_context = torch.cat((cnn_context, vit_context), dim=1) # Shape: (Batch, 1792)
        fused_context = self.dropout(fused_context)

        # 4. Multi-Task Routing
        outputs = {}
        for part_name, head_layer in self.heads.items():
            outputs[part_name] = head_layer(fused_context)

        return outputs

# Instantiate and push to GPU
model = DualStreamHydra().to(device)
print("Dual-Stream Hybrid Architecture successfully loaded onto GPU.")

Loading ConvNeXt Base Stream...
Loading ViT Stream...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Dual-Stream Hybrid Architecture successfully loaded onto GPU.


In [35]:
# Instantiate and push to GPU
model = DualStreamHydra().to(device)
print("Hydra Architecture successfully built and loaded onto GPU.")

Loading ConvNeXt Base Stream...
Loading ViT Stream...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Hydra Architecture successfully built and loaded onto GPU.


In [36]:
import torch
import numpy as np

print("Instantly calculating class weights from RAM...")

class_counts = np.zeros(4)

# 1. Bypass the image loader by accessing the raw dataframe underneath the PyTorch Subset
original_dataset = train_dataset.dataset
df = original_dataset.data_frame
train_indices = train_dataset.indices

# 2. Isolate only the rows used for training
train_df = df.iloc[train_indices]

# 3. Fast Pandas counting (No image loading!)
for part in EXPECTED_PARTS:
    counts = train_df[part].value_counts()
    for severity in range(4):
        if severity in counts:
            class_counts[severity] += counts[severity]

print(f"Total individual parts evaluated: {int(class_counts.sum())}")
print(f"Raw Count Distribution [Class 0, 1, 2, 3]: {class_counts.astype(int)}")

# 4. Apply the Inverse Class Frequency Formula
total_samples = class_counts.sum()
num_classes = 4

exact_weights = total_samples / (num_classes * (class_counts + 1e-6))
exact_weights = (exact_weights / exact_weights.sum()) * num_classes

print("\n" + "="*70)
print(f"🚀 COPY THESE TWO LINES INTO YOUR PHASE 1 AND PHASE 2 TRAINING CELLS:")
print(f"weights = torch.tensor({np.round(exact_weights, 4).tolist()}, dtype=torch.float32).to(device)")
print(f"criterion = nn.CrossEntropyLoss(weight=weights)")
print("="*70)

Instantly calculating class weights from RAM...
Total individual parts evaluated: 23472
Raw Count Distribution [Class 0, 1, 2, 3]: [17829  1368  2842  1433]

🚀 COPY THESE TWO LINES INTO YOUR PHASE 1 AND PHASE 2 TRAINING CELLS:
weights = torch.tensor([0.1221, 1.5919, 0.7663, 1.5197], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)


#Model Training

##Phase-1 Training

In [38]:
from tqdm import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

weights = torch.tensor([0.1232, 1.5892, 0.7813, 1.5063], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

EPOCHS = 15
best_val_loss = float('inf')
save_path = '/content/best_hydra_model.pth'

print("Commencing Phase 1 Training...")

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0

    # --- TRAINING PASS WITH PROGRESS BAR ---
    # Wrap the train_loader in tqdm for a live UI
    train_loop = tqdm(train_loader, leave=True, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for images, labels_dict in train_loop:
        images = images.to(device)
        targets = {part: labels.to(device) for part, labels in labels_dict.items()}

        optimizer.zero_grad()

        predictions = model(images)

        batch_loss = 0.0
        for part in EXPECTED_PARTS:
            batch_loss += criterion(predictions[part], targets[part])

        batch_loss.backward()
        optimizer.step()

        total_train_loss += batch_loss.item()

        # Update the live progress bar with the current loss
        train_loop.set_postfix(loss=batch_loss.item())

    avg_train_loss = total_train_loss / len(train_loader)

    # --- VALIDATION PASS WITH PROGRESS BAR ---
    model.eval()
    total_val_loss = 0.0

    val_loop = tqdm(val_loader, leave=True, desc=f"Epoch {epoch+1}/{EPOCHS} [Val  ]")

    with torch.no_grad():
        for images, labels_dict in val_loop:
            images = images.to(device)
            targets = {part: labels.to(device) for part, labels in labels_dict.items()}

            predictions = model(images)

            batch_loss = 0.0
            for part in EXPECTED_PARTS:
                batch_loss += criterion(predictions[part], targets[part])

            total_val_loss += batch_loss.item()
            val_loop.set_postfix(loss=batch_loss.item())

    avg_val_loss = total_val_loss / len(val_loader)

    print(f"\n--- Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} ---")

    # Save the best weights
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), save_path)
        print("  -> New best model saved to Drive!")

Commencing Phase 1 Training...


Epoch 1/15 [Val  ]: 100%|██████████| 11/11 [00:08<00:00,  1.30it/s, loss=18]



--- Epoch 1 Summary | Train Loss: 20.7308 | Val Loss: 20.5914 ---
  -> New best model saved to Drive!


Epoch 2/15 [Val  ]: 100%|██████████| 11/11 [00:07<00:00,  1.38it/s, loss=17.4]



--- Epoch 2 Summary | Train Loss: 20.0322 | Val Loss: 20.1931 ---
  -> New best model saved to Drive!


...


--- Epoch 13 Summary | Train Loss: 16.5686 | Val Loss: 17.9627 ---
  -> New best model saved to Drive!


Epoch 14/15 [Val  ]: 100%|██████████| 11/11 [00:08<00:00,  1.35it/s, loss=14]



--- Epoch 14 Summary | Train Loss: 16.3979 | Val Loss: 17.8595 ---
  -> New best model saved to Drive!


Epoch 15/15 [Val  ]: 100%|██████████| 11/11 [00:08<00:00,  1.35it/s, loss=13.9]



--- Epoch 15 Summary | Train Loss: 16.1521 | Val Loss: 17.7528 ---
  -> New best model saved to Drive!


##Phase-2 Training

In [41]:
import torch
import torch.nn as nn
from tqdm import tqdm
import shutil

print("\n" + "="*50)
print("🚀 COMMENCING PHASE 2: DUAL-STREAM FINE TUNING")
print("="*50 + "\n")

# --- 1. THE LOSS FUNCTION FIX (ANTI-ZERO-GUESSER) ---
# Replace the numbers below with the exact array generated by the RAM scanner script
weights = torch.tensor([0.1232, 1.5892, 0.7813, 1.5063], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

# --- 2. UNFREEZE THE BACKBONES ---
print("Unfreezing the final 2 layers of ConvNeXt...")
for param in model.cnn_features[-2:].parameters():
    param.requires_grad = True

print("Unfreezing the final 2 blocks of the Vision Transformer...")
for param in model.vit.encoder.layer[-2:].parameters():
    param.requires_grad = True

# --- 3. VRAM-OPTIMIZED OPTIMIZER ---
PHASE_2_LR = 1e-5

# The Fix: We filter the parameters so AdamW only tracks the layers that are actually unfreezed.
# This saves gigabytes of VRAM.
optimizer_phase_2 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE_2_LR
)

# --- 4. TRAINING SETUP ---
PHASE_2_EPOCHS = 5
phase_2_save_path = '/content/best_hydra_model_phase2.pth'

for epoch in range(PHASE_2_EPOCHS):
    model.train()
    total_train_loss = 0.0

    # --- TRAINING PASS WITH TQDM ---
    train_loop = tqdm(train_loader, leave=True, desc=f"Phase 2 Epoch {epoch+1}/{PHASE_2_EPOCHS} [Train]")

    for images, labels_dict in train_loop:
        images = images.to(device)
        targets = {part: labels.to(device) for part, labels in labels_dict.items()}

        optimizer_phase_2.zero_grad()

        predictions = model(images)

        batch_loss = 0.0
        for part in EXPECTED_PARTS:
            batch_loss += criterion(predictions[part], targets[part])

        batch_loss.backward()
        optimizer_phase_2.step()

        total_train_loss += batch_loss.item()

        # Live update the progress bar with the current batch loss
        train_loop.set_postfix(loss=batch_loss.item())

    avg_train_loss = total_train_loss / len(train_loader)

    # --- VALIDATION PASS WITH TQDM ---
    model.eval()
    total_val_loss = 0.0

    val_loop = tqdm(val_loader, leave=True, desc=f"Phase 2 Epoch {epoch+1}/{PHASE_2_EPOCHS} [Val  ]")

    with torch.no_grad():
        for images, labels_dict in val_loop:
            images = images.to(device)
            targets = {part: labels.to(device) for part, labels in labels_dict.items()}

            predictions = model(images)

            batch_loss = 0.0
            for part in EXPECTED_PARTS:
                batch_loss += criterion(predictions[part], targets[part])

            total_val_loss += batch_loss.item()

            # Live update the validation progress bar
            val_loop.set_postfix(loss=batch_loss.item())

    avg_val_loss = total_val_loss / len(val_loader)

    print(f"\n--- Phase 2 Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} ---")

    # Save the absolute best weights across both phases
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        # Just save it directly to the Colab environment
        save_path = '/content/best_hydra_model_phase2.pth'
        torch.save(model.state_dict(), save_path)

        print("  -> Phase 2 Upgrade: New best overall model saved locally!")

print(f"\n🎉 Complete pipeline finished! Final production weights saved to {save_path}")


🚀 COMMENCING PHASE 2: DUAL-STREAM FINE TUNING

Unfreezing the final 2 layers of ConvNeXt...
Unfreezing the final 2 blocks of the Vision Transformer...


Phase 2 Epoch 1/5 [Val  ]: 100%|██████████| 11/11 [00:08<00:00,  1.32it/s, loss=13.3]



--- Phase 2 Epoch 1 Summary | Train Loss: 15.3396 | Val Loss: 17.2711 ---
  -> Phase 2 Upgrade: New best overall model saved locally!


Phase 2 Epoch 2/5 [Val  ]: 100%|██████████| 11/11 [00:08<00:00,  1.34it/s, loss=13.1]



--- Phase 2 Epoch 2 Summary | Train Loss: 15.0888 | Val Loss: 17.1751 ---
  -> Phase 2 Upgrade: New best overall model saved locally!


...


--- Phase 2 Epoch 3 Summary | Train Loss: 14.9167 | Val Loss: 17.0997 ---
  -> Phase 2 Upgrade: New best overall model saved locally!


Phase 2 Epoch 4/5 [Val  ]: 100%|██████████| 11/11 [00:08<00:00,  1.37it/s, loss=13.1]



--- Phase 2 Epoch 4 Summary | Train Loss: 14.8703 | Val Loss: 17.0367 ---
  -> Phase 2 Upgrade: New best overall model saved locally!


Phase 2 Epoch 5/5 [Val  ]: 100%|██████████| 11/11 [00:08<00:00,  1.36it/s, loss=12.9]



--- Phase 2 Epoch 5 Summary | Train Loss: 14.5825 | Val Loss: 16.9033 ---
  -> Phase 2 Upgrade: New best overall model saved locally!

🎉 Complete pipeline finished! Final production weights saved to /content/best_hydra_model_phase2.pth


In [46]:
import shutil
import os

# Source path of the trained PyTorch model
source_path = '/content/best_hydra_model_phase2.pth'

# Destination directory in Google Drive
drive_dir = '/content/drive/MyDrive/'

# Create the destination directory if it doesn't exist
os.makedirs(drive_dir, exist_ok=True)

# Destination path for the model in Google Drive
destination_path = os.path.join(drive_dir, 'best_hydra_model_phase2.pth')

# Copy the file to Google Drive
shutil.copy(source_path, destination_path)

print(f"✅ Model successfully uploaded to Google Drive: {destination_path}")

✅ Model successfully uploaded to Google Drive: /content/drive/MyDrive/best_hydra_model_phase2.pth


#Saving the Model

In [ ]:
!pip install -q onnx onnxscript onnxruntime

In [ ]:
import cv2
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import convnext_base
from transformers import ViTModel, ViTConfig
from PIL import Image

# --- 1. DEFINITIONS & RULES ENGINE ---
EXPECTED_PARTS = [
    "front_bumper", "rear_bumper", "hood", "trunk", "roof",
    "windshield", "rear_window", "left_front_fender", "right_front_fender",
    "left_rear_quarter_panel", "right_rear_quarter_panel", "left_front_door",
    "right_front_door", "left_rear_door", "right_rear_door",
    "headlights", "taillights", "grille"
]

MATERIAL_LOGIC = {
    "glass": ["windshield", "rear_window", "headlights", "taillights"],
    "plastic": ["front_bumper", "rear_bumper", "grille"],
    "metal": [
        "hood", "trunk", "roof", "left_front_fender", "right_front_fender",
        "left_rear_quarter_panel", "right_rear_quarter_panel", "left_front_door",
        "right_front_door", "left_rear_door", "right_rear_door"
    ]
}

def determine_repair_action(part_name: str, severity: int) -> str:
    if severity == 0: return "No action required"
    if part_name in MATERIAL_LOGIC["glass"]: return "Replacement" if severity >= 2 else "Repair"
    if part_name in MATERIAL_LOGIC["plastic"]: return "Replacement" if severity >= 2 else "Repair"
    if part_name in MATERIAL_LOGIC["metal"]: return "Replacement" if severity == 3 else "Repair"
    return "Replacement" if severity == 3 else "Repair"

# --- 2. ARCHITECTURE ---
class DualStreamHydra(nn.Module):
    def __init__(self, num_classes=4):
        super(DualStreamHydra, self).__init__()
        convnext = convnext_base(weights=None)
        self.cnn_features = convnext.features
        self.cnn_pool = convnext.avgpool

        vit_blueprint = ViTConfig.from_pretrained("google/vit-base-patch16-224-in21k")
        self.vit = ViTModel(vit_blueprint)

        self.dropout = nn.Dropout(0.4)
        self.heads = nn.ModuleDict({
            part: nn.Linear(1792, num_classes) for part in EXPECTED_PARTS
        })

    def forward(self, x):
        cnn_out = self.cnn_features(x)
        cnn_out = self.cnn_pool(cnn_out)
        cnn_context = torch.flatten(cnn_out, 1)

        vit_out = self.vit(pixel_values=x)
        vit_context = vit_out.pooler_output

        fused_context = self.dropout(torch.cat((cnn_context, vit_context), dim=1))
        return {part: head(fused_context) for part, head in self.heads.items()}

# --- 3. INITIALIZATION ---
def load_vision_engine(weights_path):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"[*] Booting ML Engine on: {device}")

    model = DualStreamHydra()
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return model, transform, device

# --- 4. VIDEO PROCESSING LOGIC ---
def analyze_video(video_path, model, transform, device):
    print(f"[*] Analyzing video: {video_path}")
    damage_tracker = {part: {"severity": 0, "confidence": 0.0} for part in EXPECTED_PARTS}

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[!] Error: Could not open video file {video_path}")
        return None

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    if fps == 0: fps = 30

    frame_count = 0
    frames_processed = 0

    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret: break

            if frame_count % fps == 0:
                frames_processed += 1

                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                input_tensor = transform(Image.fromarray(rgb_frame)).unsqueeze(0).to(device)

                predictions = model(input_tensor)

                for part in EXPECTED_PARTS:
                    probabilities = F.softmax(predictions[part], dim=1)[0]
                    max_prob, predicted_class = torch.max(probabilities, dim=0)

                    confidence_pct = round(max_prob.item() * 100, 2)
                    severity = predicted_class.item()

                    current_max_sev = damage_tracker[part]["severity"]
                    current_max_conf = damage_tracker[part]["confidence"]

                    if severity > current_max_sev:
                        damage_tracker[part] = {"severity": severity, "confidence": confidence_pct}
                    elif severity == current_max_sev and confidence_pct > current_max_conf:
                        damage_tracker[part]["confidence"] = confidence_pct

            frame_count += 1

    cap.release()
    print(f"[*] Analysis complete. Processed {frames_processed} total keyframes.")

    final_report = {}
    for part, data in damage_tracker.items():
        severity_score = data["severity"]
        confidence = data["confidence"]

        final_report[part] = {
            "recommended_action": determine_repair_action(part, severity_score),
            "raw_severity_score": severity_score,
            "ai_confidence_score": f"{confidence}%"
        }

    return final_report

# --- 5. EXECUTION ---
if __name__ == "__main__":
    # Update these to match where the files are on your local hard drive
    WEIGHTS_FILE = "./best_hydra_model_phase2.pth"
    TEST_VIDEO = "./12676972-sd_640_360_25fps.mp4"

    try:
        engine, img_transform, compute_device = load_vision_engine(WEIGHTS_FILE)
        report = analyze_video(TEST_VIDEO, engine, img_transform, compute_device)

        if report:
            print("\n" + "="*50)
            print("🚀 FINAL DAMAGE REPORT (JSON)")
            print("="*50)
            print(json.dumps(report, indent=4))

    except Exception as e:
        print(f"\n[!] Execution failed: {e}")